In [3]:
# -*- coding: utf-8 -*-
"""
Physics-Informed Neural Network (PINN) for Slope Stability — Device 108
========================================================================
Ablation Study: PIML (Full Richards) vs Vanilla MLP (Data-Only Baseline)

  PIML: ∂θ/∂t = ∂/∂z [ K(ψ) · (∂ψ/∂z + 1) ]  ← physics constrained
  ML  : Data loss only (LAM=0, LAM_ZVAR=0)     ← pure data driven

FIXES APPLIED (PIML):
  Fix 1: θ_pred diagnostic + psi outlier clip (2nd–99th percentile)
  Fix 2: Physics loss normalization (unit mismatch সমাধান)
  Fix 3: z-gradient variance penalty (trivial satisfaction সমাধান)
"""
import warnings; warnings.filterwarnings("ignore")
import os, json, datetime
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler  # scaler_y only
from sklearn.metrics import r2_score, mean_squared_error
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── Reproducibility ─────────────────────────────────────────
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

OUT = "/content/piml_figures_108_pinn_richards"
os.makedirs(OUT, exist_ok=True)

STYLE = {
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.labelsize": 11, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
    "xtick.major.size": 3.5, "ytick.major.size": 3.5,
    "legend.frameon": False, "legend.fontsize": 9,
    "figure.dpi": 300, "savefig.dpi": 300,
    "savefig.bbox": "tight", "savefig.facecolor": "white",
}
plt.rcParams.update(STYLE)

C = {
    "rain": "#4895EF", "obs": "#E63946", "pred": "#2EC4B6",
    "train": "#3A86FF", "val": "#FF006E", "test": "#FB5607",
    "psi": "#7209B7", "u": "#F77F00", "fos": "#06D6A0",
    "warn": "#EF233C", "grid": "#CCCCCC", "resid": "#480CA8",
    "phys": "#023E8A", "vg1": "#0077B6", "vg2": "#00B4D8",
}

def savefig(fig, name):
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  → Saved: {path}")
    return path

def add_panel_label(ax, label, x=-0.08, y=1.06):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top", ha="right")

def light_grid(ax, axis="y"):
    ax.grid(axis=axis, color=C["grid"], lw=0.5, ls="--", alpha=0.7)

# ═══════════════════════════════════════════════════════════
# VAN GENUCHTEN PARAMETERS  (Sandy Clay Loam, Device 108)
# ═══════════════════════════════════════════════════════════
VG = dict(
    theta_r=0.05, theta_s=0.55,
    alpha=5.9, n=1.48, m=1 - 1/1.48,
    Ks=0.0043,           # m/h
)
VG["Ks_ms"] = VG["Ks"] / 3600.0  # m/s

# Collocation depths (m, positive downward)
Z_SENSOR    = 0.30
Z_COLLOC    = np.array([ 0.30, 0.50], dtype=np.float32)

# ═══════════════════════════════════════════════════════════
# VAN GENUCHTEN TORCH UTILITIES
# ═══════════════════════════════════════════════════════════

def vg_Se_t(theta, vg):
    tc = theta.clamp(vg["theta_r"] + 1e-6, vg["theta_s"] - 1e-6)
    return (tc - vg["theta_r"]) / (vg["theta_s"] - vg["theta_r"])

def vg_psi_t(theta, vg):
    Se = vg_Se_t(theta, vg).clamp(1e-6, 1 - 1e-6)
    h  = (1.0 / vg["alpha"]) * (Se**(-1.0 / vg["m"]) - 1.0)**(1.0 / vg["n"])
    return -h

def vg_K_t(theta, vg):
    Se = vg_Se_t(theta, vg).clamp(1e-6, 1 - 1e-6)
    Kr = Se**0.5 * (1.0 - (1.0 - Se**(1.0 / vg["m"]))**vg["m"])**2
    return vg["Ks_ms"] * Kr

# NumPy equivalents (evaluation only)
def vg_Se_np(theta, vg):
    tc = np.clip(theta, vg["theta_r"]+1e-6, vg["theta_s"]-1e-6)
    return (tc - vg["theta_r"]) / (vg["theta_s"] - vg["theta_r"])

def vg_psi_np(theta, vg):
    Se = np.clip(vg_Se_np(theta, vg), 1e-6, 1-1e-6)
    return -(1.0/vg["alpha"]) * (Se**(-1.0/vg["m"]) - 1.0)**(1.0/vg["n"])

def vg_Kr_np(theta, vg):
    Se = np.clip(vg_Se_np(theta, vg), 1e-6, 1-1e-6)
    return Se**0.5 * (1-(1-Se**(1/vg["m"]))**vg["m"])**2

def vg_K_np(theta, vg):
    return vg["Ks_ms"] * vg_Kr_np(theta, vg)

# ═══════════════════════════════════════════════════════════
# 1-2.  LOAD & PREPROCESS  (Device 107 only)
# ═══════════════════════════════════════════════════════════
print("\n[Step 1-2] Loading data for Device 108 ...")
df_raw = pd.read_csv("/content/pinn_108_new.csv", low_memory=False)
df_raw = df_raw.dropna(subset=["timestamp", "devID", "soil", "rain"])
df_raw["devID"]     = df_raw["devID"].astype(int)
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], dayfirst=False, errors="coerce")
df_raw = df_raw.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

d108 = df_raw[df_raw["devID"] == 108].copy().reset_index(drop=True)
d108["t_min"] = d108["timestamp"].dt.floor("min")
df = d108[["t_min","soil","rain"]].rename(
    columns={"t_min":"timestamp","soil":"theta","rain":"rain"}).copy()
df = df.sort_values("timestamp").reset_index(drop=True)
df["t_sec"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()

df = df.dropna(subset=["rain","theta","t_sec"]).reset_index(drop=True)
df = df.iloc[::2].reset_index(drop=True)
print(f"  Rows after subsample: {len(df):,}")

LAG = 15
n   = len(df)
n_tr  = int(0.80 * n)
n_val = int(0.90 * n)

df_tr  = df.iloc[:n_tr].copy().reset_index(drop=True)
df_val = df.iloc[n_tr:n_val].copy().reset_index(drop=True)
df_te  = df.iloc[n_val:].copy().reset_index(drop=True)

ROLL_W = 5
for _d in [df_tr, df_val, df_te]:
    _d["theta"] = _d["theta"].rolling(ROLL_W, center=False, min_periods=1).median()

def build_features(d, lag=1):
    X_rows, y_rows = [], []
    for i in range(lag, len(d)):
        lags = [d["theta"].iloc[i-k] for k in range(1, lag+1)]
        X_rows.append([d["rain"].iloc[i]] + lags)
        y_rows.append(d["theta"].iloc[i])
    X      = np.array(X_rows, dtype=np.float32)
    y      = np.array(y_rows, dtype=np.float32).reshape(-1,1)
    rain_  = d["rain"].values[lag:].astype(np.float32)
    theta_ = d["theta"].values[lag:].astype(np.float32)
    t_sec_ = d["t_sec"].values[lag:].astype(np.float32)
    ts_    = d["timestamp"].values[lag:]
    return X, y, rain_, theta_, t_sec_, ts_

X_tr_raw,  y_tr_raw,  rain_tr,  theta_tr,  t_sec_tr,  ts_tr  = build_features(df_tr,  LAG)
X_val_raw, y_val_raw, rain_val, theta_val, t_sec_val, ts_val  = build_features(df_val, LAG)
X_te_raw,  y_te_raw,  rain_te,  theta_te,  t_sec_te,  ts_te   = build_features(df_te,  LAG)

# ── Physics-based normalization (replaces MinMaxScaler for X) ──────
# Rain normalized by regional design storm threshold (RAIN_MAX=75 mm/30min);
# lagged θ bounded by Van Genuchten theta_r / theta_s — valid beyond observed range.
RAIN_MAX = 75.0   # mm/30 min — regional upper bound (IDF curve)

def physics_normalize_X(X_raw):
    """Physics-based normalization: rain → RAIN_MAX; lagged θ → VG bounds."""
    X_norm = X_raw.copy()
    X_norm[:, 0]  = X_raw[:, 0] / RAIN_MAX                                           # rain
    X_norm[:, 1:] = (X_raw[:, 1:] - VG["theta_r"]) / (VG["theta_s"] - VG["theta_r"])  # lags
    return X_norm.astype(np.float32)

scaler_y = MinMaxScaler().fit(y_tr_raw)  # output scaler unchanged

X_tr  = physics_normalize_X(X_tr_raw)
X_val = physics_normalize_X(X_val_raw)
X_te  = physics_normalize_X(X_te_raw)
y_tr  = scaler_y.transform(y_tr_raw).astype(np.float32)
y_val = scaler_y.transform(y_val_raw).astype(np.float32)
y_te  = scaler_y.transform(y_te_raw).astype(np.float32)

X_np      = np.concatenate([X_tr_raw, X_val_raw, X_te_raw], axis=0)
theta_np  = np.concatenate([theta_tr,  theta_val,  theta_te],  axis=0)
rain_np   = np.concatenate([rain_tr,   rain_val,   rain_te],   axis=0)
t_sec_np  = np.concatenate([t_sec_tr,  t_sec_val,  t_sec_te],  axis=0)
ts_all_np = np.concatenate([ts_tr,     ts_val,     ts_te],     axis=0)
X_norm    = physics_normalize_X(X_np)

T_MIN   = float(t_sec_tr.min())
T_MAX   = float(t_sec_tr.max()) + 1e-8
Z_MAX   = float(Z_COLLOC.max()) + 1e-8

def to_t(a): return torch.tensor(a, dtype=torch.float32).to(DEVICE)

Xt, yt   = to_t(X_tr),  to_t(y_tr)
Xv, yv   = to_t(X_val), to_t(y_val)
Xte, yte = to_t(X_te),  to_t(y_te)

rain_t  = to_t(rain_tr[:, None])
t_sec_t = to_t(t_sec_tr[:, None])
dt_tr   = np.diff(t_sec_tr, prepend=t_sec_tr[0]).clip(min=1.0)
dt_t    = to_t(dt_tr[:, None])

# ═══════════════════════════════════════════════════════════
# 3.  MODEL
# ═══════════════════════════════════════════════════════════
print("\n[Step 3] Building PINN (true autograd Richards) ...")

class RichardsPINN(nn.Module):
    def __init__(self, feat_dim, h=128):
        super().__init__()
        in_dim = feat_dim + 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, h), nn.Tanh(),
            nn.Linear(h, h),      nn.Tanh(),
            nn.Linear(h, h),      nn.Tanh(),
            nn.Linear(h, h//2),   nn.Tanh(),
            nn.Linear(h//2, 1),   nn.Sigmoid(),
        )
        self.theta_lo = VG["theta_r"] + 0.01
        self.theta_hi = VG["theta_s"] - 0.01

    def forward(self, z_norm, t_norm, feat):
        x   = torch.cat([z_norm, t_norm, feat], dim=1)
        raw = self.net(x)
        theta = self.theta_lo + (self.theta_hi - self.theta_lo) * raw
        return theta

model = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ═══════════════════════════════════════════════════════════
# 4.  PHYSICS LOSS  — [FIX 3] dθ/dz penalty added
# ═══════════════════════════════════════════════════════════

def richards_residual(model, z_norm, t_norm, feat):
    """
    Compute 1D Richards PDE residual + dθ/dz for z-gradient penalty.

    Returns
    -------
    residual   : (B,1)  m³/m³/s
    theta      : (B,1)  predicted θ̂
    dtheta_dz  : (B,1)  ∂θ/∂z  [FIX 3: returned for z-gradient penalty]
    """
    theta = model(z_norm, t_norm, feat)

    dtheta_dt = torch.autograd.grad(
        outputs=theta, inputs=t_norm,
        grad_outputs=torch.ones_like(theta),
        create_graph=True, retain_graph=True
    )[0]

    psi = vg_psi_t(theta, VG)

    dpsi_dz = torch.autograd.grad(
        outputs=psi, inputs=z_norm,
        grad_outputs=torch.ones_like(psi),
        create_graph=True, retain_graph=True
    )[0]

    K = vg_K_t(theta, VG)

    dpsi_dz_phys = dpsi_dz / Z_MAX
    flux = K * (dpsi_dz_phys + 1.0)

    dflux_dz = torch.autograd.grad(
        outputs=flux, inputs=z_norm,
        grad_outputs=torch.ones_like(flux),
        create_graph=True, retain_graph=True
    )[0]

    dflux_dz_phys = dflux_dz / Z_MAX
    dtheta_dt_phys = dtheta_dt / (T_MAX - T_MIN)
    residual = dtheta_dt_phys - dflux_dz_phys

    # ── [FIX 3] ∂θ/∂z — for z-gradient variance penalty ─────
    dtheta_dz = torch.autograd.grad(
        outputs=theta, inputs=z_norm,
        grad_outputs=torch.ones_like(theta),
        create_graph=True, retain_graph=True
    )[0]

    return residual, theta, dtheta_dz   # ← now returns 3 values


# ═══════════════════════════════════════════════════════════
# 5.  COLLOCATION POINT BUILDER
# ═══════════════════════════════════════════════════════════

def make_colloc_batch(t_sec_batch, rain_batch, feat_batch):
    Nz = len(Z_COLLOC)
    B  = t_sec_batch.shape[0]
    z_phys   = np.tile(Z_COLLOC, B).reshape(-1, 1).astype(np.float32)
    z_norm_c = z_phys / Z_MAX
    t_phys_c = np.repeat(t_sec_batch, Nz).reshape(-1, 1).astype(np.float32)
    t_norm_c = (t_phys_c - T_MIN) / (T_MAX - T_MIN)
    feat_c   = np.repeat(feat_batch, Nz, axis=0).astype(np.float32)
    z_c    = torch.tensor(z_norm_c, dtype=torch.float32,
                          device=DEVICE, requires_grad=True)
    t_c    = torch.tensor(t_norm_c, dtype=torch.float32,
                          device=DEVICE, requires_grad=True)
    feat_c = torch.tensor(feat_c,   dtype=torch.float32, device=DEVICE)
    return z_c, t_c, feat_c


# ═══════════════════════════════════════════════════════════
# 6.  TRAINING LOOP
# ═══════════════════════════════════════════════════════════
print("\n[Step 4-6] Training PINN with true Richards autograd loss ...")
EPOCHS   = 600
LR       = 5e-4
LAM      = 5
BATCH    = 512
WARMUP   = 100
LAM_ZVAR = 0.1   # [FIX 3] z-gradient variance penalty weight

# ── [FIX 2] Physical scale for residual normalization ───────
THETA_RANGE = VG["theta_s"] - VG["theta_r"]          # 0.50 m³/m³
T_RANGE_SEC = float(T_MAX - T_MIN)                    # seconds
DTHDT_SCALE = THETA_RANGE / T_RANGE_SEC               # typical ∂θ/∂t, m³/m³/s
print(f"  ∂θ/∂t scale for normalization: {DTHDT_SCALE:.4e} m³/m³/s")

y_min_t = torch.tensor(scaler_y.data_min_[0], dtype=torch.float32, device=DEVICE)
y_scl_t = torch.tensor(scaler_y.scale_[0],    dtype=torch.float32, device=DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return float(epoch + 1) / float(WARMUP)
    progress = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
mse_loss  = nn.MSELoss()

z_sens_norm = torch.tensor([[Z_SENSOR / Z_MAX]], dtype=torch.float32, device=DEVICE)

t_tr_norm   = ((t_sec_tr  - T_MIN) / (T_MAX - T_MIN)).astype(np.float32)
t_tr_norm_t = to_t(t_tr_norm[:, None])

dataset = TensorDataset(Xt, yt, rain_t, t_tr_norm_t, dt_t,
                        to_t(t_sec_tr[:, None]))
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=False)

data_losses  = []
phys_losses  = []
train_losses = []
val_losses   = []
lr_history   = []

best_val   = np.inf
best_state = None

z_v_rep  = z_sens_norm.expand(Xv.shape[0], -1).detach()
t_v_norm = to_t(((t_sec_val - T_MIN) / (T_MAX - T_MIN))[:, None])

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_data  = 0.0
    ep_phys  = 0.0

    for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
        optimizer.zero_grad()
        B = Xb.shape[0]

        # ── Data loss at sensor depth ─────────────────────────
        z_d = z_sens_norm.expand(B, -1).requires_grad_(True)
        t_d = tb_norm.clone().requires_grad_(True)
        theta_pred = model(z_d, t_d, Xb)

        pred_norm = (theta_pred - y_min_t) * y_scl_t
        loss_d = mse_loss(pred_norm, yb)

        # ── Physics loss at collocation depths ────────────────
        tb_sec_np = tb_sec.detach().cpu().numpy().flatten()
        Xb_np     = Xb.detach().cpu().numpy()
        z_c, t_c, feat_c = make_colloc_batch(tb_sec_np, None, Xb_np)

        # [FIX 3] richards_residual now returns 3 values
        resid, _, dtheta_dz = richards_residual(model, z_c, t_c, feat_c)

        # ── [FIX 2] Normalize residual → dimensionless ────────
        resid_norm = resid / (DTHDT_SCALE + 1e-12)
        loss_p = (resid_norm ** 2).mean()

        # ── [FIX 3] Z-gradient variance penalty ───────────────
        # Reshape to (B, Nz), penalize if all depths have same dθ/dz
        Nz = len(Z_COLLOC)
        dtheta_dz_r = dtheta_dz.view(B, Nz)          # (B, Nz)
        z_var_loss  = -dtheta_dz_r.var(dim=1).mean()  # maximize z-variance

        loss = loss_d + LAM * loss_p + LAM_ZVAR * z_var_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        ep_data += loss_d.item()
        ep_phys += loss_p.item()

    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    model.eval()
    with torch.no_grad():
        vl = mse_loss(
            (model(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t,
            yv
        ).item()

    n_batches = len(loader)
    data_losses.append(ep_data / n_batches)
    phys_losses.append(ep_phys / n_batches)
    train_losses.append((ep_data + LAM * ep_phys) / n_batches)
    val_losses.append(vl)
    lr_history.append(current_lr)

    if vl < best_val:
        best_val   = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 100 == 0:
        # ── [FIX 3 diagnostic] λ·Phys / Data ratio ──────────
        ratio = (LAM * phys_losses[-1]) / (data_losses[-1] + 1e-15)
        print(f"  Epoch {epoch:4d} | Data {data_losses[-1]:.5f} | "
              f"Phys(norm) {phys_losses[-1]:.5f} | λ·Phys/Data ratio: {ratio:.3f} | "
              f"Val {vl:.5f} | LR {current_lr:.2e}")

model.load_state_dict(best_state)
print(f"\n  Best val: {best_val:.5f}  @ epoch {int(np.argmin(val_losses))+1}")

def predict_sensor(X_norm_np, t_sec_arr):
    model.eval()
    N = X_norm_np.shape[0]
    z_s  = z_sens_norm.expand(N, -1)
    t_s  = to_t(((t_sec_arr - T_MIN) / (T_MAX - T_MIN))[:, None])
    Xn   = to_t(X_norm_np)
    with torch.no_grad():
        return model(z_s, t_s, Xn).cpu().numpy().flatten()

# ═══════════════════════════════════════════════════════════
# MODEL SAVE
# ═══════════════════════════════════════════════════════════
print("\n[Model Save] Saving model ...")
model_path = os.path.join(OUT, "piml_slope_model_108_pinn_richards.pt")
torch.save({
    "model_state_dict"  : best_state,
    "model_architecture": {"class": "RichardsPINN", "feat_dim": X_tr.shape[1], "hidden": 128},
    "governing_equation": (
        "1D Full Richards: ∂θ/∂t = ∂/∂z[K(ψ(θ))(∂ψ/∂z + 1)]\n"
        "All derivatives via torch.autograd.grad — no approximation."
    ),
    "hyperparameters": {
        "epochs": EPOCHS, "lr": LR, "lambda_phys": LAM,
        "lambda_zvar": LAM_ZVAR,
        "batch_size": BATCH, "lag": LAG,
        "z_colloc_m": Z_COLLOC.tolist(), "z_sensor_m": Z_SENSOR,
        "dthdt_scale": DTHDT_SCALE,
    },
    "scalers": {
        "rain_max": RAIN_MAX,
        "vg_theta_r": VG["theta_r"],
        "vg_theta_s": VG["theta_s"],
        "scaler_y_min": scaler_y.data_min_.tolist(),
        "scaler_y_scale": scaler_y.scale_.tolist(),
        "T_MIN": T_MIN, "T_MAX": T_MAX, "Z_MAX": Z_MAX,
    },
    "vg_params": VG,
    "slope_params": dict(slope_deg=41.0, c_kPa=5.0, phi_deg=28.0, gamma_s=18.5, H_m=1.5),
    "device_id": 108,
    "train_losses": train_losses, "val_losses": val_losses,
    "data_losses": data_losses,   "phys_losses": phys_losses,
    "lr_history": lr_history,
    "best_val_loss": best_val,
    "timestamp": datetime.datetime.now().isoformat(),
}, model_path)
print(f"  → Model saved: {model_path}")

# ═══════════════════════════════════════════════════════════════
# ██  ABLATION — ML BASELINE (LAM=0, LAM_ZVAR=0, Data-Only)  ██
# ═══════════════════════════════════════════════════════════════
print("\n" + "█"*65)
print("  ABLATION: Training ML Baseline (Vanilla MLP, Data-Only)")
print("  LAM=0  LAM_ZVAR=0  — No physics constraint")
print("█"*65)

# ── Re-init same architecture from scratch (same seed) ──────
torch.manual_seed(42); np.random.seed(42)
model_ml = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
print(f"  ML Baseline Parameters: {sum(p.numel() for p in model_ml.parameters()):,}")

optimizer_ml = optim.AdamW(model_ml.parameters(), lr=LR, weight_decay=1e-4)
scheduler_ml = optim.lr_scheduler.LambdaLR(optimizer_ml, lr_lambda)

data_losses_ml  = []
train_losses_ml = []
val_losses_ml   = []

best_val_ml   = np.inf
best_state_ml = None

EPOCHS_ML = EPOCHS  # same number of epochs for fair comparison

for epoch in range(1, EPOCHS_ML + 1):
    model_ml.train()
    ep_data_ml = 0.0

    for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
        optimizer_ml.zero_grad()
        B = Xb.shape[0]

        # ── Data loss ONLY (no physics, no z-grad) ─────────────
        z_d = z_sens_norm.expand(B, -1)          # no requires_grad needed
        t_d = tb_norm.clone()                     # no requires_grad needed
        theta_pred_ml = model_ml(z_d, t_d, Xb)

        pred_norm_ml = (theta_pred_ml - y_min_t) * y_scl_t
        loss_ml = mse_loss(pred_norm_ml, yb)     # ONLY data loss

        loss_ml.backward()
        torch.nn.utils.clip_grad_norm_(model_ml.parameters(), 1.0)
        optimizer_ml.step()
        ep_data_ml += loss_ml.item()

    scheduler_ml.step()

    model_ml.eval()
    with torch.no_grad():
        vl_ml = mse_loss(
            (model_ml(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t,
            yv
        ).item()

    n_batches = len(loader)
    data_losses_ml.append(ep_data_ml / n_batches)
    train_losses_ml.append(ep_data_ml / n_batches)
    val_losses_ml.append(vl_ml)

    if vl_ml < best_val_ml:
        best_val_ml   = vl_ml
        best_state_ml = {k: v.clone() for k, v in model_ml.state_dict().items()}

    if epoch % 100 == 0:
        print(f"  [ML] Epoch {epoch:4d} | Data {data_losses_ml[-1]:.5f} | "
              f"Val {vl_ml:.5f} | LR {optimizer_ml.param_groups[0]['lr']:.2e}")

model_ml.load_state_dict(best_state_ml)
best_ep_ml = int(np.argmin(val_losses_ml)) + 1
print(f"\n  [ML] Best val: {best_val_ml:.5f}  @ epoch {best_ep_ml}")

def predict_sensor_ml(X_norm_np, t_sec_arr):
    model_ml.eval()
    N = X_norm_np.shape[0]
    z_s  = z_sens_norm.expand(N, -1)
    t_s  = to_t(((t_sec_arr - T_MIN) / (T_MAX - T_MIN))[:, None])
    Xn   = to_t(X_norm_np)
    with torch.no_grad():
        return model_ml(z_s, t_s, Xn).cpu().numpy().flatten()


# ═══════════════════════════════════════════════════════════════
# 7.  EVALUATE — PIML (Physics-Informed)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 7a] Evaluate PIML ...")
theta_pred_tr  = predict_sensor(X_tr,   t_sec_tr)
theta_pred_val = predict_sensor(X_val,  t_sec_val)
theta_pred_te  = predict_sensor(X_te,   t_sec_te)
theta_pred_all = predict_sensor(X_norm, t_sec_np)
theta_obs_all  = theta_np.copy()

r2_tr  = r2_score(theta_tr,  theta_pred_tr)
r2_val = r2_score(theta_val, theta_pred_val)
r2_te  = r2_score(theta_te,  theta_pred_te)
rmse_te = np.sqrt(mean_squared_error(theta_te, theta_pred_te))
residuals = theta_te - theta_pred_te

print(f"  [PIML] Train R²: {r2_tr:.4f}  |  Val R²: {r2_val:.4f}  |  Test R²: {r2_te:.4f}")
print(f"  [PIML] Test RMSE: {rmse_te:.5f} m³/m³")

rain_all = rain_np.copy()
ts       = pd.to_datetime(ts_all_np)

# ═══════════════════════════════════════════════════════════════
# 7b. EVALUATE — ML Baseline (Data-Only)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 7b] Evaluate ML Baseline ...")
theta_ml_tr  = predict_sensor_ml(X_tr,   t_sec_tr)
theta_ml_val = predict_sensor_ml(X_val,  t_sec_val)
theta_ml_te  = predict_sensor_ml(X_te,   t_sec_te)
theta_ml_all = predict_sensor_ml(X_norm, t_sec_np)

r2_ml_tr   = r2_score(theta_tr,  theta_ml_tr)
r2_ml_val  = r2_score(theta_val, theta_ml_val)
r2_ml_te   = r2_score(theta_te,  theta_ml_te)
rmse_ml_te = np.sqrt(mean_squared_error(theta_te, theta_ml_te))
residuals_ml = theta_te - theta_ml_te

print(f"  [ML]   Train R²: {r2_ml_tr:.4f}  |  Val R²: {r2_ml_val:.4f}  |  Test R²: {r2_ml_te:.4f}")
print(f"  [ML]   Test RMSE: {rmse_ml_te:.5f} m³/m³")


# ═══════════════════════════════════════════════════════════════
# 8.  Van Genuchten θ→ψ  — BOTH MODELS
# ═══════════════════════════════════════════════════════════════
print("\n[Step 8] Van Genuchten θ→ψ (both models) ...")

# ── PIML ──────────────────────────────────────────────────────
frac_low = (theta_pred_all < 0.10).mean()
psi_all_raw = vg_psi_np(theta_pred_all, VG)
psi_lo  = np.percentile(psi_all_raw, 2)
psi_all = np.clip(psi_all_raw, psi_lo, 0.0)
K_all   = vg_K_np(theta_pred_all, VG)
print(f"  [PIML] θ outlier frac (<0.10): {frac_low:.4f}")
print(f"  [PIML] |ψ| clipped range: [{-psi_all.max():.3f}, {-psi_all.min():.3f}] m")

# ── ML Baseline ───────────────────────────────────────────────
frac_low_ml = (theta_ml_all < 0.10).mean()
psi_ml_raw  = vg_psi_np(theta_ml_all, VG)
psi_lo_ml   = np.percentile(psi_ml_raw, 2)
psi_ml_all  = np.clip(psi_ml_raw, psi_lo_ml, 0.0)
K_ml_all    = vg_K_np(theta_ml_all, VG)
print(f"  [ML]   θ outlier frac (<0.10): {frac_low_ml:.4f}")
print(f"  [ML]   |ψ| clipped range: [{-psi_ml_all.max():.3f}, {-psi_ml_all.min():.3f}] m")

theta_vg = np.linspace(VG["theta_r"]+0.001, VG["theta_s"]-0.001, 300)
psi_vg   = vg_psi_np(theta_vg, VG)
kr_vg    = vg_Kr_np(theta_vg, VG)

# ═══════════════════════════════════════════════════════════════
# 9.  Pore pressure & FoS — BOTH MODELS
# ═══════════════════════════════════════════════════════════════
gamma_w = 9.81
SLOPE = 25.0; beta = np.radians(SLOPE)
c_ = 5.0; phi_ = np.radians(28.0)
gamma_s = 18.5; H = 1.5
sigma_n = gamma_s * H * np.cos(beta)**2
tau_d   = gamma_s * H * np.sin(beta) * np.cos(beta)

# PIML FoS
u_all   = gamma_w * psi_all
FoS_all = (c_ + np.maximum(sigma_n - u_all, 0) * np.tan(phi_)) / (tau_d + 1e-8)

# ML FoS
u_ml    = gamma_w * psi_ml_all
FoS_ml  = (c_ + np.maximum(sigma_n - u_ml, 0) * np.tan(phi_)) / (tau_d + 1e-8)

print(f"\n  [PIML] FoS min={FoS_all.min():.3f}  mean={FoS_all.mean():.3f}  FoS<1={(FoS_all<1).sum()}")
print(f"  [ML]   FoS min={FoS_ml.min():.3f}  mean={FoS_ml.mean():.3f}   FoS<1={(FoS_ml<1).sum()}")

# ═══════════════════════════════════════════════════════════════
# 10. Richards Residual — PIML (exact autograd) + ML (effective)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 10] Computing Richards residual ...")

# PIML residual via autograd
N_all  = X_norm.shape[0]
X_all_t = to_t(X_norm)
model.eval()
CHUNK = 1024
resid_list = []
for i in range(0, N_all, CHUNK):
    z_c = torch.full((min(CHUNK, N_all-i), 1), Z_SENSOR/Z_MAX,
                     dtype=torch.float32, device=DEVICE, requires_grad=True)
    t_c = to_t(((t_sec_np[i:i+CHUNK] - T_MIN)/(T_MAX-T_MIN))[:,None])
    t_c.requires_grad_(True)
    f_c = X_all_t[i:i+CHUNK]
    resid_c, _, _ = richards_residual(model, z_c, t_c, f_c)
    resid_list.append(resid_c.detach().cpu().numpy().flatten())

phys_residual = np.concatenate(resid_list)
mu_r = phys_residual.mean(); sd_r = phys_residual.std()
print(f"  [PIML] Richards residual  μ = {mu_r:.4e}  σ = {sd_r:.4e}  m³/m³/s")

# ML "effective" Richards residual (how much it violates physics)
resid_ml_list = []
model_ml.eval()
for i in range(0, N_all, CHUNK):
    z_c = torch.full((min(CHUNK, N_all-i), 1), Z_SENSOR/Z_MAX,
                     dtype=torch.float32, device=DEVICE, requires_grad=True)
    t_c = to_t(((t_sec_np[i:i+CHUNK] - T_MIN)/(T_MAX-T_MIN))[:,None])
    t_c.requires_grad_(True)
    f_c = X_all_t[i:i+CHUNK]
    resid_c, _, _ = richards_residual(model_ml, z_c, t_c, f_c)
    resid_ml_list.append(resid_c.detach().cpu().numpy().flatten())

phys_residual_ml = np.concatenate(resid_ml_list)
mu_r_ml = phys_residual_ml.mean(); sd_r_ml = phys_residual_ml.std()
print(f"  [ML]   Richards residual  μ = {mu_r_ml:.4e}  σ = {sd_r_ml:.4e}  m³/m³/s")


# ═══════════════════════════════════════════════════════════════
# ██████████  F I G U R E S  (Ablation Study)  ████████████████
# ═══════════════════════════════════════════════════════════════
print("\n[Step 11] Generating ablation study figures ...")

n_tr_plot  = len(theta_tr)
n_val_plot = n_tr_plot + len(theta_val)
epochs_arr = np.arange(1, EPOCHS + 1)
best_ep    = int(np.argmin(val_losses)) + 1

# colour shorthands
C_PIML = C["pred"]   # teal  — PIML
C_ML   = C["warn"]   # red   — ML baseline

# ──────────────────────────────────────────────────────────────
# FIG 1 — Loss Convergence Comparison (PIML vs ML)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(11, 12),
                         gridspec_kw={"hspace": 0.45})

ax = axes[0]
ax.semilogy(epochs_arr, train_losses,    color=C_PIML, lw=1.8, label="PIML train (data+physics)")
ax.semilogy(epochs_arr, val_losses,      color=C_PIML, lw=1.4, ls="--", alpha=0.8, label="PIML val")
ax.semilogy(epochs_arr, train_losses_ml, color=C_ML,   lw=1.8, label="ML train (data only)")
ax.semilogy(epochs_arr, val_losses_ml,   color=C_ML,   lw=1.4, ls="--", alpha=0.8, label="ML val")
ax.axvline(best_ep, color="gray", lw=1.0, ls=":", label=f"PIML best ep ({best_ep})")
ax.axvline(best_ep_ml, color="orange", lw=1.0, ls=":", label=f"ML best ep ({best_ep_ml})")
ax.set_ylabel("MSE Loss"); light_grid(ax)
ax.set_title("(A)  Train / Validation Loss — PIML vs ML Baseline", loc="left", fontweight="bold")
ax.legend(fontsize=8, ncol=2)

ax = axes[1]
ax.semilogy(epochs_arr, np.array(phys_losses), color=C["phys"], lw=1.8,
            label="PIML physics loss (normalised Richards PDE)")
ax.semilogy(epochs_arr, LAM * np.array(phys_losses), color=C["phys"], lw=1.2,
            ls="--", alpha=0.6, label=f"λ·Physics (λ={LAM})")
ax.axhline(1.0, color="gray", lw=0.8, ls=":", alpha=0.5, label="Reference = 1.0")
ax.set_ylabel("Richards PDE Loss (−)"); light_grid(ax)
ax.set_title("(B)  Physics Loss Convergence (PIML only)", loc="left", fontweight="bold")
ax.legend(fontsize=8)

ax = axes[2]
data_arr = np.array(data_losses)
data_ml_arr = np.array(data_losses_ml)
ax.semilogy(epochs_arr, data_arr,    color=C_PIML, lw=1.8, label="PIML data component")
ax.semilogy(epochs_arr, data_ml_arr, color=C_ML,   lw=1.8, label="ML data loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Data MSE Loss"); light_grid(ax)
ax.set_title("(C)  Data Loss Component — PIML vs ML", loc="left", fontweight="bold")
ax.legend(fontsize=8)
ax.text(0.98, 0.05,
        f"PIML best val: {min(val_losses):.2e}\nML best val:   {min(val_losses_ml):.2e}",
        transform=ax.transAxes, fontsize=9, ha="right",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))

fig.suptitle(
    f"Training Convergence — Ablation: PIML (λ={LAM}, λ_zvar={LAM_ZVAR}) vs Vanilla MLP (λ=0)\n"
    f"Same architecture · Same data · Same epochs ({EPOCHS}) · AdamW · Cosine LR",
    fontsize=11, fontweight="bold"
)
savefig(fig, "fig01_loss_convergence_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 2 — Scatter R² : PIML vs ML side by side (6 panels)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

splits_piml = [
    ("PIML — Train", theta_tr,  theta_pred_tr,  r2_tr,     C_PIML),
    ("PIML — Val",   theta_val, theta_pred_val, r2_val,    C_PIML),
    ("PIML — Test",  theta_te,  theta_pred_te,  r2_te,     C_PIML),
]
splits_ml = [
    ("ML — Train", theta_tr,  theta_ml_tr,  r2_ml_tr,  C_ML),
    ("ML — Val",   theta_val, theta_ml_val, r2_ml_val, C_ML),
    ("ML — Test",  theta_te,  theta_ml_te,  r2_ml_te,  C_ML),
]

for row, splits in enumerate([splits_piml, splits_ml]):
    for col, (label, obs, pred, r2, col_c) in enumerate(splits):
        ax = axes[row][col]
        ax.scatter(obs, pred, s=6, alpha=0.3, color=col_c, rasterized=True)
        lim = [min(obs.min(), pred.min())-0.002, max(obs.max(), pred.max())+0.002]
        ax.plot(lim, lim, "k--", lw=1.2, label="1:1")
        m, b = np.polyfit(obs, pred, 1)
        xfit = np.linspace(*lim, 100)
        ax.plot(xfit, m*xfit+b, color=col_c, lw=1.4, alpha=0.8, label="OLS")
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("θ Observed (m³/m³)")
        ax.set_ylabel("θ Predicted (m³/m³)")
        rmse_s = np.sqrt(mean_squared_error(obs, pred))
        ax.set_title(f"{label}\nR²={r2:.4f}  RMSE={rmse_s:.4f}")
        ax.legend(fontsize=7); light_grid(ax)

fig.suptitle("Ablation Study — Scatter R²: PIML (top row) vs Vanilla MLP (bottom row)\n"
             "Device 108 · Full Richards Autograd · Same Architecture",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
savefig(fig, "fig02_scatter_r2_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 3 — θ Time-series Comparison (Test window)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios":[1, 2.5, 2.5], "hspace": 0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)"); light_grid(axes[0])
axes[0].set_title("A — Rainfall Intensity", fontweight="bold", loc="left")

axes[1].plot(ts, theta_obs_all,  color=C["obs"],  lw=1.3, alpha=0.9, label="θ Observed", zorder=3)
axes[1].plot(ts, theta_pred_all, color=C_PIML,    lw=1.3, alpha=0.9,
             label=f"PIML (R²={r2_te:.4f})", ls="--", zorder=2)
tr_end  = ts[n_tr_plot-1]; val_end = ts[n_val_plot-1]
axes[1].axvspan(ts[0], tr_end,   alpha=0.05, color=C["train"], label="Train")
axes[1].axvspan(tr_end, val_end, alpha=0.08, color=C["val"],   label="Val")
axes[1].axvspan(val_end, ts[-1], alpha=0.08, color=C["test"],  label="Test")
axes[1].set_ylabel("θ PIML (m³/m³)")
axes[1].legend(ncol=2, fontsize=8); light_grid(axes[1])
axes[1].set_title(f"B — PIML Prediction  [Test R²={r2_te:.4f}  RMSE={rmse_te:.4f}]",
                  fontweight="bold", loc="left")

axes[2].plot(ts, theta_obs_all, color=C["obs"], lw=1.3, alpha=0.9, label="θ Observed", zorder=3)
axes[2].plot(ts, theta_ml_all,  color=C_ML,     lw=1.3, alpha=0.9,
             label=f"ML Baseline (R²={r2_ml_te:.4f})", ls="--", zorder=2)
axes[2].axvspan(ts[0], tr_end,   alpha=0.05, color=C["train"])
axes[2].axvspan(tr_end, val_end, alpha=0.08, color=C["val"])
axes[2].axvspan(val_end, ts[-1], alpha=0.08, color=C["test"])
axes[2].set_ylabel("θ ML (m³/m³)"); axes[2].set_xlabel("Timestamp")
axes[2].legend(ncol=2, fontsize=8); light_grid(axes[2])
axes[2].set_title(f"C — ML Baseline Prediction  [Test R²={r2_ml_te:.4f}  RMSE={rmse_ml_te:.4f}]",
                  fontweight="bold", loc="left")

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)

fig.suptitle("θ Time-Series: PIML vs Vanilla MLP — Device 108", fontsize=12, fontweight="bold")
savefig(fig, "fig03_theta_timeseries_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 4 — Residual Comparison (Test set)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
ts_te_plot = ts[n_val_plot:]

# PIML residuals over time
axes[0][0].axhline(0, color="black", lw=0.8)
axes[0][0].fill_between(ts_te_plot, residuals, 0, where=(residuals>=0),
                        color=C_PIML, alpha=0.5, label="Over-pred")
axes[0][0].fill_between(ts_te_plot, residuals, 0, where=(residuals<0),
                        color=C["phys"], alpha=0.5, label="Under-pred")
axes[0][0].set_title("PIML Residuals vs Time (Test)"); axes[0][0].legend(fontsize=8)
axes[0][0].set_ylabel("θ_obs − θ_pred (m³/m³)"); light_grid(axes[0][0])
for t in axes[0][0].get_xticklabels(): t.set_rotation(15)

# PIML residual histogram
axes[0][1].hist(residuals, bins=40, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.85)
axes[0][1].axvline(0, color="black", lw=1.0, ls="--")
axes[0][1].axvline(residuals.mean(), color=C_ML, lw=1.4, ls="--",
                   label=f"μ={residuals.mean():.4f}")
axes[0][1].axvline(residuals.std(), color="gray", lw=1.2, ls=":",
                   label=f"σ={residuals.std():.4f}")
axes[0][1].axvline(-residuals.std(), color="gray", lw=1.2, ls=":")
axes[0][1].set_title("PIML Residual Distribution (Test)")
axes[0][1].set_xlabel("Residual (m³/m³)"); axes[0][1].legend(fontsize=8); light_grid(axes[0][1])

# ML residuals over time
axes[1][0].axhline(0, color="black", lw=0.8)
axes[1][0].fill_between(ts_te_plot, residuals_ml, 0, where=(residuals_ml>=0),
                        color=C_ML, alpha=0.5, label="Over-pred")
axes[1][0].fill_between(ts_te_plot, residuals_ml, 0, where=(residuals_ml<0),
                        color=C["u"], alpha=0.5, label="Under-pred")
axes[1][0].set_title("ML Baseline Residuals vs Time (Test)"); axes[1][0].legend(fontsize=8)
axes[1][0].set_ylabel("θ_obs − θ_pred (m³/m³)"); axes[1][0].set_xlabel("Timestamp")
light_grid(axes[1][0])
for t in axes[1][0].get_xticklabels(): t.set_rotation(15)

# ML residual histogram
axes[1][1].hist(residuals_ml, bins=40, color=C_ML, edgecolor="white", lw=0.3, alpha=0.85)
axes[1][1].axvline(0, color="black", lw=1.0, ls="--")
axes[1][1].axvline(residuals_ml.mean(), color=C_PIML, lw=1.4, ls="--",
                   label=f"μ={residuals_ml.mean():.4f}")
axes[1][1].axvline(residuals_ml.std(), color="gray", lw=1.2, ls=":",
                   label=f"σ={residuals_ml.std():.4f}")
axes[1][1].axvline(-residuals_ml.std(), color="gray", lw=1.2, ls=":")
axes[1][1].set_title("ML Baseline Residual Distribution (Test)")
axes[1][1].set_xlabel("Residual (m³/m³)"); axes[1][1].legend(fontsize=8); light_grid(axes[1][1])

fig.suptitle("Residual Analysis — PIML (top) vs Vanilla MLP (bottom) — Test Set  Dev108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig04_residuals_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 5 — Van Genuchten (same as original — PIML only)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(-psi_vg, theta_vg, color=C["vg1"], lw=2.0, label="VG Model")
psi_te_abs = np.abs(vg_psi_np(theta_pred_te, VG))
sort_idx   = np.argsort(psi_te_abs)
axes[0].scatter(psi_te_abs[sort_idx[::5]], theta_pred_te[sort_idx[::5]],
                s=6, alpha=0.3, color=C_PIML, rasterized=True, label="PIML θ̂ (test)")
psi_ml_te_abs = np.abs(vg_psi_np(theta_ml_te, VG))
sort_idx_ml   = np.argsort(psi_ml_te_abs)
axes[0].scatter(psi_ml_te_abs[sort_idx_ml[::5]], theta_ml_te[sort_idx_ml[::5]],
                s=6, alpha=0.2, color=C_ML, rasterized=True, label="ML θ̂ (test)")
axes[0].set_xlabel("|ψ| Matric Suction (m)"); axes[0].set_ylabel("θ (m³/m³)")
axes[0].set_title("(a) Soil Water Characteristic — PIML vs ML"); axes[0].set_xscale("log")
axes[0].legend(fontsize=8); light_grid(axes[0])
axes[0].text(0.98, 0.06,
             f"θ_r={VG['theta_r']} θ_s={VG['theta_s']}\nα={VG['alpha']} n={VG['n']}",
             transform=axes[0].transAxes, fontsize=8, ha="right",
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))

axes[1].plot(theta_vg, kr_vg, color=C["psi"], lw=2.0)
axes[1].fill_between(theta_vg, kr_vg, alpha=0.15, color=C["psi"])
axes[1].set_xlabel("θ (m³/m³)"); axes[1].set_ylabel("K_r (−)")
axes[1].set_title("(b) Relative Hydraulic Conductivity (VG)"); light_grid(axes[1])
fig.suptitle("Van Genuchten Curves — Dev108", fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig05_vg_characteristic_curves.png")

# ──────────────────────────────────────────────────────────────
# FIG 6 — Matric Suction Comparison (PIML vs ML)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True,
                         gridspec_kw={"height_ratios":[1,2,2,2], "hspace":0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)")
axes[0].set_title("Matric Suction: PIML vs ML — Dev10  [ψ clipped 2–99th pct]", loc="left")
light_grid(axes[0])

axes[1].plot(ts, theta_pred_all, color=C_PIML, lw=1.1, alpha=0.9, label="PIML θ̂")
axes[1].plot(ts, theta_ml_all,   color=C_ML,   lw=1.1, alpha=0.7, label="ML θ̂", ls="--")
axes[1].plot(ts, theta_obs_all,  color=C["obs"], lw=0.8, alpha=0.5, ls=":", label="Observed")
axes[1].set_ylabel("θ (m³/m³)"); axes[1].legend(fontsize=8, ncol=3); light_grid(axes[1])

axes[2].plot(ts, -psi_all,    color=C_PIML, lw=1.4, label="PIML |ψ|")
axes[2].fill_between(ts, -psi_all, alpha=0.10, color=C_PIML)
axes[2].set_ylabel("|ψ| PIML (m)"); axes[2].legend(fontsize=8); light_grid(axes[2])

axes[3].plot(ts, -psi_ml_all, color=C_ML,   lw=1.4, label="ML |ψ|")
axes[3].fill_between(ts, -psi_ml_all, alpha=0.10, color=C_ML)
axes[3].set_ylabel("|ψ| ML (m)"); axes[3].set_xlabel("Timestamp")
axes[3].legend(fontsize=8); light_grid(axes[3])

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig06_matric_suction_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 7 — FoS Comparison Time-series
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True,
                         gridspec_kw={"height_ratios":[1, 2, 2.5, 2.5], "hspace":0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)")
axes[0].set_title("Factor of Safety: PIML vs ML Baseline — Dev108", loc="left")
light_grid(axes[0])

axes[1].plot(ts, u_all,  color=C_PIML, lw=1.3, label="PIML pore pressure u")
axes[1].plot(ts, u_ml,   color=C_ML,   lw=1.3, label="ML pore pressure u", ls="--", alpha=0.7)
axes[1].axhline(0, color="gray", lw=0.7, ls=":")
axes[1].set_ylabel("u (kPa)"); axes[1].legend(fontsize=8, ncol=2); light_grid(axes[1])

axes[2].plot(ts, FoS_all, color=C_PIML, lw=1.4, label=f"PIML FoS  (min={FoS_all.min():.3f})")
axes[2].axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS=1.0 ⚠")
axes[2].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warning")
axes[2].fill_between(ts, FoS_all, 1.0, where=(FoS_all<1.0),
                     color=C["warn"], alpha=0.35, label=f"Failure (n={(FoS_all<1.0).sum()})")
axes[2].set_ylabel("FoS — PIML (−)"); axes[2].legend(ncol=2, fontsize=8); light_grid(axes[2])
axes[2].set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

axes[3].plot(ts, FoS_ml, color=C_ML, lw=1.4, label=f"ML FoS  (min={FoS_ml.min():.3f})")
axes[3].axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS=1.0 ⚠")
axes[3].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warning")
axes[3].fill_between(ts, FoS_ml, 1.0, where=(FoS_ml<1.0),
                     color=C["warn"], alpha=0.35, label=f"Failure (n={(FoS_ml<1.0).sum()})")
axes[3].set_ylabel("FoS — ML (−)"); axes[3].set_xlabel("Timestamp")
axes[3].legend(ncol=2, fontsize=8); light_grid(axes[3])
axes[3].set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig07_fos_timeseries_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 8 — FoS Distribution Comparison
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

all_fos = np.concatenate([FoS_all, FoS_ml])
bins    = np.linspace(all_fos.min()-0.05, all_fos.max()+0.05, 60)

axes[0].hist(FoS_all, bins=bins, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.75,
             label=f"PIML  (μ={FoS_all.mean():.3f}, min={FoS_all.min():.3f})")
axes[0].hist(FoS_ml,  bins=bins, color=C_ML,   edgecolor="white", lw=0.3, alpha=0.55,
             label=f"ML    (μ={FoS_ml.mean():.3f}, min={FoS_ml.min():.3f})")
axes[0].axvline(1.0, color="black",       lw=2.0, ls="--", label="FoS=1.0 ⚠")
axes[0].axvline(1.3, color="darkorange",  lw=1.5, ls=":",  label="FoS=1.3 Warning")
axes[0].set_xlabel("FoS (−)"); axes[0].set_ylabel("Count")
axes[0].set_title("(a) FoS Histogram — PIML vs ML"); axes[0].legend(fontsize=8); light_grid(axes[0])

sorted_piml = np.sort(FoS_all); exc_piml = 1 - np.arange(1, len(sorted_piml)+1)/len(sorted_piml)
sorted_ml   = np.sort(FoS_ml);  exc_ml   = 1 - np.arange(1, len(sorted_ml)+1)/len(sorted_ml)
axes[1].semilogy(sorted_piml, exc_piml, color=C_PIML, lw=2.0, label="PIML")
axes[1].semilogy(sorted_ml,   exc_ml,   color=C_ML,   lw=2.0, label="ML Baseline", ls="--")
axes[1].axvline(1.0, color="black",      lw=1.5, ls="--", label="FoS=1.0")
axes[1].axvline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3")
axes[1].set_xlabel("FoS (−)"); axes[1].set_ylabel("Exceedance Prob.")
axes[1].set_title("(b) FoS Exceedance Probability"); axes[1].legend(fontsize=8); light_grid(axes[1])

fig.suptitle("Factor of Safety Distribution: PIML vs Vanilla MLP — Dev108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig08_fos_distribution_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 9 — Richards PDE Residual: PIML vs ML
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# PIML residual time-series
axes[0][0].plot(ts, phys_residual, color=C_PIML, lw=0.8, alpha=0.85)
axes[0][0].axhline(0, color="gray", lw=0.7, ls=":")
axes[0][0].fill_between(ts, phys_residual, 0,
                         where=(np.abs(phys_residual) > 2*sd_r),
                         color=C["warn"], alpha=0.4, label=f">2σ  σ={sd_r:.2e}")
axes[0][0].set_title("PIML — Richards Residual ∂θ/∂t − ∇·flux  (m³/m³/s)")
axes[0][0].set_ylabel("Residual (m³/m³/s)"); axes[0][0].legend(fontsize=8); light_grid(axes[0][0])
axes[0][0].text(0.99, 0.97, f"μ={mu_r:.2e}\nσ={sd_r:.2e}",
                transform=axes[0][0].transAxes, fontsize=8, va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))
for t in axes[0][0].get_xticklabels(): t.set_rotation(15)

# PIML residual histogram
axes[0][1].hist(phys_residual, bins=50, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.85)
axes[0][1].axvline(0, color="black", lw=1.0, ls="--")
axes[0][1].axvline( sd_r, color="gray", lw=1.2, ls=":", label=f"±σ={sd_r:.2e}")
axes[0][1].axvline(-sd_r, color="gray", lw=1.2, ls=":")
axes[0][1].set_title("PIML — PDE Residual Distribution")
axes[0][1].set_xlabel("Residual (m³/m³/s)"); axes[0][1].legend(fontsize=8); light_grid(axes[0][1])

# ML residual time-series
axes[1][0].plot(ts, phys_residual_ml, color=C_ML, lw=0.8, alpha=0.85)
axes[1][0].axhline(0, color="gray", lw=0.7, ls=":")
axes[1][0].fill_between(ts, phys_residual_ml, 0,
                         where=(np.abs(phys_residual_ml) > 2*sd_r_ml),
                         color=C["warn"], alpha=0.4, label=f">2σ  σ={sd_r_ml:.2e}")
axes[1][0].set_title("ML Baseline — Effective Richards Residual (m³/m³/s)")
axes[1][0].set_ylabel("Residual (m³/m³/s)"); axes[1][0].set_xlabel("Timestamp")
axes[1][0].legend(fontsize=8); light_grid(axes[1][0])
axes[1][0].text(0.99, 0.97, f"μ={mu_r_ml:.2e}\nσ={sd_r_ml:.2e}",
                transform=axes[1][0].transAxes, fontsize=8, va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))
for t in axes[1][0].get_xticklabels(): t.set_rotation(15)

# ML residual histogram
axes[1][1].hist(phys_residual_ml, bins=50, color=C_ML, edgecolor="white", lw=0.3, alpha=0.85)
axes[1][1].axvline(0, color="black", lw=1.0, ls="--")
axes[1][1].axvline( sd_r_ml, color="gray", lw=1.2, ls=":", label=f"±σ={sd_r_ml:.2e}")
axes[1][1].axvline(-sd_r_ml, color="gray", lw=1.2, ls=":")
axes[1][1].set_title("ML Baseline — PDE Residual Distribution")
axes[1][1].set_xlabel("Residual (m³/m³/s)"); axes[1][1].legend(fontsize=8); light_grid(axes[1][1])

fig.suptitle(
    "Richards PDE Residual: PIML (physics-constrained, top) vs ML Baseline (unconstrained, bottom)\n"
    "Smaller σ → better physical consistency  |  PIML should have σ ≪ ML",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
savefig(fig, "fig09_pde_residual_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 10 — Full Dashboard (PIML — same as original)
# ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 17))
gs  = gridspec.GridSpec(5, 1, figure=fig, hspace=0.50)
ax0 = fig.add_subplot(gs[0])
ax0.fill_between(ts, rain_all, color=C["rain"], alpha=0.75, step="mid")
ax0.set_ylabel("Rainfall (mm/min)")
ax0.set_title("A — Rainfall Intensity", fontweight="bold", loc="left")
light_grid(ax0); add_panel_label(ax0, "(A)", y=1.08)

ax1 = fig.add_subplot(gs[1])
ax1.plot(ts, theta_obs_all,  color=C["obs"],  lw=1.2, alpha=0.85, label="θ Observed")
ax1.plot(ts, theta_pred_all, color=C_PIML,    lw=1.2, alpha=0.85, label="θ PIML-Richards", ls="--")
ax1.plot(ts, theta_ml_all,   color=C_ML,      lw=1.0, alpha=0.65, label="θ ML-Baseline",  ls=":")
ax1.set_ylabel("θ (m³/m³)")
ax1.set_title(f"B — Soil Moisture  [PIML R²={r2_te:.3f}  ML R²={r2_ml_te:.3f}]",
              fontweight="bold", loc="left")
ax1.legend(ncol=3, fontsize=8); light_grid(ax1); add_panel_label(ax1, "(B)", y=1.08)

ax2 = fig.add_subplot(gs[2])
ax2.plot(ts, -psi_all,    color=C_PIML, lw=1.3, label="PIML |ψ|")
ax2.plot(ts, -psi_ml_all, color=C_ML,   lw=1.1, alpha=0.7, label="ML |ψ|", ls="--")
ax2.fill_between(ts, -psi_all, alpha=0.08, color=C_PIML)
ax2.set_ylabel("|ψ| clipped (m)")
ax2.set_title("C — Matric Suction (VG, clipped)", fontweight="bold", loc="left")
ax2.legend(fontsize=8); light_grid(ax2); add_panel_label(ax2, "(C)", y=1.08)

ax3 = fig.add_subplot(gs[3])
ax3.plot(ts, u_all, color=C_PIML, lw=1.3, label="PIML u")
ax3.plot(ts, u_ml,  color=C_ML,   lw=1.1, alpha=0.7, label="ML u", ls="--")
ax3.axhline(0, color="gray", lw=0.7, ls=":")
ax3.fill_between(ts, u_all, 0, where=(u_all>0), color=C["warn"], alpha=0.2, label="Positive u")
ax3.set_ylabel("u Pore Pressure (kPa)")
ax3.set_title("D — Pore Water Pressure", fontweight="bold", loc="left")
ax3.legend(fontsize=8, ncol=3); light_grid(ax3); add_panel_label(ax3, "(D)", y=1.08)

ax4 = fig.add_subplot(gs[4])
ax4.plot(ts, FoS_all, color=C_PIML, lw=1.4,
         label=f"PIML FoS (min={FoS_all.min():.3f})")
ax4.plot(ts, FoS_ml,  color=C_ML,   lw=1.2, alpha=0.75, ls="--",
         label=f"ML FoS   (min={FoS_ml.min():.3f})")
ax4.axhline(1.0, color=C["warn"],    lw=1.8, ls="--", label="FoS=1.0 ⚠ Failure")
ax4.axhline(1.3, color="darkorange", lw=1.2, ls=":",  label="FoS=1.3 Warning")
ax4.fill_between(ts, FoS_all, 1.0, where=(FoS_all<1.0), color=C["warn"], alpha=0.35)
ax4.set_ylabel("FoS (−)"); ax4.set_xlabel("Timestamp")
ax4.set_title("E — Factor of Safety (Infinite Slope)", fontweight="bold", loc="left")
ax4.legend(ncol=2, fontsize=8); light_grid(ax4); add_panel_label(ax4, "(E)", y=1.08)
ax4.set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

for ax in [ax0, ax1, ax2, ax3, ax4]:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)

fig.suptitle(
    "Full Dashboard — PIML (Richards, Autograd) vs ML Baseline (Data-Only)\n"
    f"Device 108  |  λ_phys={LAM}  λ_zvar={LAM_ZVAR}  |  PIML R²={r2_te:.4f}  ML R²={r2_ml_te:.4f}",
    fontsize=12, fontweight="bold", y=1.002)
savefig(fig, "fig10_full_dashboard_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 11 — ABLATION TABLE (publication-ready matplotlib table)
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.axis("off")

col_labels = ["Metric", "PIML\n(Full Richards)", "ML Baseline\n(Data-Only)", "Δ / Winner"]

def delta_str(piml_val, ml_val, higher_better=True):
    """Return improvement string."""
    diff = piml_val - ml_val
    pct  = abs(diff / (ml_val + 1e-12)) * 100
    if higher_better:
        winner = "✅ PIML" if diff > 0 else "❌ ML"
    else:
        winner = "✅ PIML" if diff < 0 else "❌ ML"
    return f"{winner}  ({abs(diff):.4f}, {pct:.1f}%)"

row_data = [
    ["Train R²",
     f"{r2_tr:.4f}", f"{r2_ml_tr:.4f}",
     delta_str(r2_tr, r2_ml_tr, higher_better=True)],

    ["Val R²",
     f"{r2_val:.4f}", f"{r2_ml_val:.4f}",
     delta_str(r2_val, r2_ml_val, higher_better=True)],

    ["Test R²",
     f"{r2_te:.4f}", f"{r2_ml_te:.4f}",
     delta_str(r2_te, r2_ml_te, higher_better=True)],

    ["Test RMSE (m³/m³)",
     f"{rmse_te:.5f}", f"{rmse_ml_te:.5f}",
     delta_str(rmse_te, rmse_ml_te, higher_better=False)],

    ["PDE Residual μ (m³/m³/s)",
     f"{mu_r:.3e}", f"{mu_r_ml:.3e}",
     delta_str(abs(mu_r), abs(mu_r_ml), higher_better=False)],

    ["PDE Residual σ (m³/m³/s)",
     f"{sd_r:.3e}", f"{sd_r_ml:.3e}",
     delta_str(sd_r, sd_r_ml, higher_better=False)],

    ["FoS Mean",
     f"{FoS_all.mean():.3f}", f"{FoS_ml.mean():.3f}",
     "—"],

    ["FoS Min",
     f"{FoS_all.min():.3f}", f"{FoS_ml.min():.3f}",
     delta_str(FoS_all.min(), FoS_ml.min(), higher_better=True)],

    ["FoS < 1.0 events",
     f"{(FoS_all<1.0).sum()}", f"{(FoS_ml<1.0).sum()}",
     "Lower = fewer false alarms"],

    ["FoS < 1.3 events",
     f"{(FoS_all<1.3).sum()}", f"{(FoS_ml<1.3).sum()}",
     "Lower = fewer warnings"],

    ["θ outlier frac (< 0.10)",
     f"{frac_low:.4f}", f"{frac_low_ml:.4f}",
     delta_str(frac_low, frac_low_ml, higher_better=False)],

    ["Physics constraint",
     "✅ Richards PDE (LAM=5)", "❌ None (LAM=0)", "PIML physically consistent"],

    ["Architecture",
     "RichardsPINN (same)", "RichardsPINN (same)", "Identical — fair comparison"],

    ["Training epochs",
     f"{EPOCHS}", f"{EPOCHS_ML}", "Identical"],
]

tbl = ax.table(
    cellText=row_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    colWidths=[0.28, 0.20, 0.20, 0.32],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1, 2.1)

# Header styling
for j in range(4):
    tbl[0, j].set_facecolor("#023E8A")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

# Row alternating + highlight key rows
highlight_rows = {2, 3, 5}   # Test R², RMSE, PDE σ — most important
for i in range(1, len(row_data)+1):
    for j in range(4):
        if i in highlight_rows:
            tbl[i, j].set_facecolor("#E3F2FD")
        elif i % 2 == 0:
            tbl[i, j].set_facecolor("#F8F9FA")
        else:
            tbl[i, j].set_facecolor("white")
        if j == 3:
            tbl[i, j].set_text_props(color="#023E8A" if "✅ PIML" in str(row_data[i-1][j]) else
                                      "#E63946"      if "❌ ML"   in str(row_data[i-1][j]) else
                                      "#333333")

ax.set_title(
    f"Table 1 — Ablation Study Results: PIML vs Vanilla MLP Baseline\n"
    f"Device 108 · Sandy Clay Loam · Slope 25° · Full Richards Equation (Autograd)\n"
    f"PIML: LAM={LAM}, LAM_ZVAR={LAM_ZVAR}   |   ML Baseline: LAM=0, LAM_ZVAR=0",
    fontsize=11, fontweight="bold", pad=18
)
plt.tight_layout()
savefig(fig, "fig11_ablation_table.png")

# ──────────────────────────────────────────────────────────────
# FINAL RESULTS SUMMARY (both models)
# ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  ABLATION STUDY — FINAL RESULTS (Dev108)")
print("  PIML: Full 1D Richards  vs  ML: Vanilla MLP (Data-Only)")
print("="*70)
print(f"  {'Metric':<30} {'PIML':>12} {'ML Baseline':>14}")
print(f"  {'-'*58}")
print(f"  {'Train R²':<30} {r2_tr:>12.4f} {r2_ml_tr:>14.4f}")
print(f"  {'Val   R²':<30} {r2_val:>12.4f} {r2_ml_val:>14.4f}")
print(f"  {'Test  R²':<30} {r2_te:>12.4f} {r2_ml_te:>14.4f}")
print(f"  {'Test  RMSE (m³/m³)':<30} {rmse_te:>12.5f} {rmse_ml_te:>14.5f}")
print(f"  {'PDE Residual μ':<30} {mu_r:>12.4e} {mu_r_ml:>14.4e}")
print(f"  {'PDE Residual σ':<30} {sd_r:>12.4e} {sd_r_ml:>14.4e}")
print(f"  {'FoS Mean':<30} {FoS_all.mean():>12.3f} {FoS_ml.mean():>14.3f}")
print(f"  {'FoS Min':<30} {FoS_all.min():>12.3f} {FoS_ml.min():>14.3f}")
print(f"  {'FoS < 1.0 events':<30} {(FoS_all<1.0).sum():>12d} {(FoS_ml<1.0).sum():>14d}")
print(f"  {'FoS < 1.3 events':<30} {(FoS_all<1.3).sum():>12d} {(FoS_ml<1.3).sum():>14d}")
print(f"  {'θ outlier frac':<30} {frac_low:>12.4f} {frac_low_ml:>14.4f}")
print("="*70)

piml_wins = sum([
    r2_te > r2_ml_te,
    rmse_te < rmse_ml_te,
    abs(sd_r) < abs(sd_r_ml),
    FoS_all.min() > FoS_ml.min(),
])
print(f"\n  PIML wins on {piml_wins}/4 key metrics (R², RMSE, PDE σ, FoS min)")
print(f"\n✅ Figures → {OUT}/")
print(f"   fig01 — Loss convergence ablation")
print(f"   fig02 — Scatter R² ablation (6 panels)")
print(f"   fig03 — θ time-series ablation")
print(f"   fig04 — Residuals ablation")
print(f"   fig05 — VG characteristic curves")
print(f"   fig06 — Matric suction ablation")
print(f"   fig07 — FoS time-series ablation")
print(f"   fig08 — FoS distribution ablation")
print(f"   fig09 — PDE residual ablation")
print(f"   fig10 — Full dashboard ablation")
print(f"   fig11 — Ablation TABLE (publication-ready)")


Device: cpu

[Step 1-2] Loading data for Device 108 ...
  Rows after subsample: 30,977

[Step 3] Building PINN (true autograd Richards) ...
  Parameters: 43,777

[Step 4-6] Training PINN with true Richards autograd loss ...
  ∂θ/∂t scale for normalization: 8.7802e-08 m³/m³/s
  Epoch  100 | Data 0.00295 | Phys(norm) 0.00002 | λ·Phys/Data ratio: 0.029 | Val 0.00052 | LR 5.00e-04
  Epoch  200 | Data 0.00229 | Phys(norm) 0.00002 | λ·Phys/Data ratio: 0.043 | Val 0.00056 | LR 4.52e-04
  Epoch  300 | Data 0.00018 | Phys(norm) 0.00002 | λ·Phys/Data ratio: 0.490 | Val 0.00004 | LR 3.27e-04
  Epoch  400 | Data 0.00009 | Phys(norm) 0.00000 | λ·Phys/Data ratio: 0.238 | Val 0.00002 | LR 1.73e-04
  Epoch  500 | Data 0.00006 | Phys(norm) 0.00000 | λ·Phys/Data ratio: 0.048 | Val 0.00002 | LR 4.77e-05
  Epoch  600 | Data 0.00006 | Phys(norm) 0.00000 | λ·Phys/Data ratio: 0.025 | Val 0.00002 | LR 0.00e+00

  Best val: 0.00002  @ epoch 419

[Model Save] Saving model ...
  → Model saved: /content/piml_figu